# 4. Dashboard visuals (pipeline step 9)

This notebook runs **pipeline step 9** (`9_dashboard_visuals`). It **prebuilds all dashboard visualization artifacts** (BupaR, DTW, FP-Growth) on **EC2** and **saves them to S3** for **direct dashboard integration**. The dashboard loads these prebuilt assets from S3 (the API returns only URLs; no computation at request time). Visuals are **SHAP/FFA-driven**: model data and feature lists come from Step 3b / 7 / 8 so process mining and itemset mining use only important features.

**Flow:** Run after [3_model_train_shap_ffa.ipynb](3_model_train_shap_ffa.ipynb). Then run [5_build_and_deploy.ipynb](5_build_and_deploy.ipynb) once to build and deploy.

**Prerequisites:**
- **Primary**: Step 3b feature importance (`3a_feature_importance/outputs/{cohort}/{age_band}/cohort_feature_importance.csv`)
- **Fallback**: Notebook 3 combined importance (`10_risk_dashboard/outputs/{cohort}/{age_band}/combined_importance.csv`)
- If neither is available, allowed_codes will be empty and visualizations may fail or use all codes.

## Steps

1. **Setup** – Resolve paths (scripts in `9_dashboard_visuals/`; outputs under `10_risk_dashboard/visualizations/{bupar,dtw,fpgrowth}/`).
2. **Feature importance heatmaps** – Aggregated and combined heatmaps for the dashboard **Feature Importance** tab; saved to `3a_feature_importance/outputs/{cohort}/plots/{cohort}_aggregated_fi_heatmap.png` and `3a_feature_importance/outputs/plots/combined_cohorts_feature_importance_heatmap.png`. Notebook 5 (deploy) expects these paths and syncs them to S3.
3. **BupaR** – Process mining sequences and plots (SHAP/FFA allowed codes when available); **uploaded to the dashboard bucket** under `{S3_DASHBOARD_PREFIX}/bupar/{cohort}/{age_band}/plots/`.
4. **DTW** – Trajectory features and plots **based on SHAP/FFA important codes** (same as BupaR/FP-Growth); plot PNGs are **uploaded to the dashboard bucket** under `{S3_DASHBOARD_PREFIX}/dtw/{cohort}/{age_band}/plots/`. The DTW tab includes **appointments vs no appointments** visuals: **Routine vs No Routine (Outcomes)** and **High-Risk vs Low-Risk Trajectories** (outcome rate by trajectory intensity / archetype). These visuals use the **full pipeline (2016–2019)**: model_events (Step 4) and DTW features are built from all years 2016–2019, not a single year. **Extreme-density cohorts** (optional) support the same routine vs no routine comparison for high-utilizer subgroups—see optional step below.

5. **FP-Growth** – Itemsets, rules, **Plotly network HTML**, and PNGs; **uploaded to the dashboard bucket** (`S3_DASHBOARD_BUCKET`, e.g. jerome-dixon.io) under `{S3_DASHBOARD_PREFIX}/fpgrowth/{cohort}/{age_band}/plots/` (e.g. `vcu/pgx-risk-calculator/fpgrowth/...`). The dashboard loads the **network plot by cohort** from these URLs.

Idempotent. Run from repo root. Prerequisites: notebook 5 done (`4_model_data`, `7_shap_analysis`, `8_ffa_analysis`); R and bupaR for BupaR.


6. **Cohort PGx** – VIP reports (PharmGKB) and network topology per cohort/age_band; outputs under `10_risk_dashboard/visualizations/cohort_pgx/`. The **PGx Cohort** dashboard tab calls GET /visualizations/cohort_pgx and displays the network iframe. Deploy syncs `cohort_pgx/` to S3.

7. **Model performance metrics and cohort metadata** – Prebuilt via `generate_metrics.py` and `generate_metadata.py` (no recomputation). Deploy (5_build_and_deploy) uploads to the dashboard bucket: `metadata/model_performance_metrics.json` (Documentation tab) and `metadata/opioid_ed.json`, `metadata/non_opioid_ed.json` (dropdowns). Frontend loads these same-origin; Lambda GET /metrics and GET /metadata are fallbacks.

8. **API** – Returns URLs to prebuilt S3 assets only (no server-side computation for visuals). Lambda GET /visualizations/cohort_pgx serves the PGx Cohort tab.

In [11]:
# Setup: paths (outputs under 10_risk_dashboard/visualizations/)
import sys
import os
import subprocess
from pathlib import Path

REPO_ROOT = Path.cwd()
if (REPO_ROOT / "py_helpers").exists():
    pass  # already repo root
else:
    for p in REPO_ROOT.parents:
        if (p / "py_helpers").exists():
            REPO_ROOT = p
            break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from py_helpers.env_utils import get_data_root, get_model_data_root

# Data root (with fallback to local nvme or Windows pgx_data)
DATA_ROOT = get_data_root()
MODEL_DATA_ROOT = get_model_data_root()
S3_BUCKET = os.environ.get("PGX_S3_BUCKET", "pgxdatalake")

# Creation code (step 9); outputs go to 10_risk_dashboard/visualizations
STEP9_ROOT = REPO_ROOT / "9_dashboard_visuals"
VISUAL_ROOT = REPO_ROOT / "10_risk_dashboard" / "visualizations"
BUPAR_VISUALS_SCRIPT = STEP9_ROOT / "bupar" / "create_bupar_visuals.py"
DTW_TRAJECTORIES_SCRIPT = STEP9_ROOT / "dtw" / "create_dtw_trajectories.py"
DTW_FEATURES_SCRIPT = STEP9_ROOT / "dtw" / "create_dtw_features.py"
DTW_VISUALS_SCRIPT = STEP9_ROOT / "dtw" / "create_dtw_visuals.py"
FPGROWTH_VISUALS_SCRIPT = STEP9_ROOT / "fpgrowth" / "create_fpgrowth_visuals.py"

print(f"Repo root: {REPO_ROOT}")
print(f"Data root (NVMe/local): {DATA_ROOT}")
print(f"Model data root: {MODEL_DATA_ROOT}")
print(f"S3 bucket: {S3_BUCKET}")
print(f"Step 9 (scripts): {STEP9_ROOT}")
print(f"Outputs: {VISUAL_ROOT}")
print(f"Plots (PNG/HTML) appear under: {VISUAL_ROOT}/bupar/outputs/<cohort>/<age_band>/plots/ (and dtw/, fpgrowth/) — run the cells below to generate them.")

Repo root: /home/pgx3874/pgx-analysis
Data root (NVMe/local): /mnt/nvme
Model data root: /mnt/nvme/4_model_data
S3 bucket: pgxdatalake
Step 9 (scripts): /home/pgx3874/pgx-analysis/9_dashboard_visuals
Outputs: /home/pgx3874/pgx-analysis/10_risk_dashboard/visualizations
Plots (PNG/HTML) appear under: /home/pgx3874/pgx-analysis/10_risk_dashboard/visualizations/bupar/outputs/<cohort>/<age_band>/plots/ (and dtw/, fpgrowth/) — run the cells below to generate them.


## Config: cohorts and age bands

Defaults match **run_dashboard_visuals.py**: all cohorts and all age bands (from REQUIRED_COHORTS), one worker per (cohort, age_band) combo (capped by CPU), no dry run. Leave `COHORTS_TO_RUN` and `AGE_BANDS_TO_RUN` empty for full pipeline; set either to limit scope.

In [12]:
from py_helpers.constants import COHORT_NAMES, AGE_BANDS

try:
    from py_helpers.constants import REQUIRED_COHORTS
except ImportError:
    _all_bands = ['0-12', '13-24', '25-44', '45-54', '55-64', '65-74', '75-84', '85-114']
    REQUIRED_COHORTS = {"opioid_ed": _all_bands, "non_opioid_ed": _all_bands}

COHORTS_TO_RUN = []
AGE_BANDS_TO_RUN = []

# Default: pipeline (cohort, age_band) from REQUIRED_COHORTS (both cohorts use full age bands 0-12 through 85-114)
if not COHORTS_TO_RUN and not AGE_BANDS_TO_RUN:
    combinations = [(c, ab) for c, bands in REQUIRED_COHORTS.items() for ab in bands]
    print("Using pipeline-supported cohort/age_band (REQUIRED_COHORTS)")
else:
    if not COHORTS_TO_RUN:
        COHORTS_TO_RUN = COHORT_NAMES.copy()
    if not AGE_BANDS_TO_RUN:
        AGE_BANDS_TO_RUN = AGE_BANDS.copy()
    combinations = [(c, ab) for c in COHORTS_TO_RUN for ab in AGE_BANDS_TO_RUN]

print(f"Cohorts: {COHORTS_TO_RUN if COHORTS_TO_RUN else list(REQUIRED_COHORTS.keys())}")
print(f"Age bands: {AGE_BANDS_TO_RUN if AGE_BANDS_TO_RUN else 'per-cohort (REQUIRED_COHORTS)'}")
print(f"Total: {len(combinations)} combinations")

# Idempotent: skip when output exists. Set FORCE_RERUN=True to pass --force and re-run all (BupaR, FP-Growth, DTW).
# FP-Growth: --force re-creates itemsets and plots. DTW: --force ignores pipeline checkpoint and plots.
FORCE_RERUN = True
# Parallel workers: one per (cohort, age_band) combo, capped by CPU (matches run_dashboard_visuals.py default).
_ncpu = getattr(os, "cpu_count", lambda: 4)() or 4
PARALLEL_WORKERS = min(_ncpu, len(combinations))
FPGROWTH_WORKERS = min(_ncpu, len(combinations))

# Generate allowed_codes JSON files if missing (uses Step 3b or notebook 3 combined_importance.csv)
from py_helpers.shap_ffa_fpgrowth_utils import write_shap_ffa_allowed_codes_for_bupar
BUPAR_OUTPUTS = REPO_ROOT / "10_risk_dashboard" / "visualizations" / "bupar" / "outputs"
BUPAR_OUTPUTS.mkdir(parents=True, exist_ok=True)

print("\n" + "="*80)
print("Generating allowed_codes JSON files from feature importance data...")
print("="*80)
print(f"Repo root: {REPO_ROOT}")
print(f"Data root: {DATA_ROOT}")
print(f"Looking for:")
print(f"  - Step 3b: 3a_feature_importance/outputs/{{cohort}}/{{age_band}}/cohort_feature_importance.csv")
print(f"  - Notebook 3: 10_risk_dashboard/outputs/{{cohort}}/{{age_band}}/combined_importance.csv")
print()

generated = 0
skipped = 0
failed = 0
for cohort_name, age_band in combinations:
    age_band_fname = age_band.replace("-", "_")
    allowed_path = BUPAR_OUTPUTS / f"allowed_codes_shap_ffa_{cohort_name}_{age_band_fname}.json"
    if allowed_path.exists():
        skipped += 1
        continue
    
    # Check if source exists
    combined_path = REPO_ROOT / "10_risk_dashboard" / "outputs" / cohort_name / age_band_fname / "combined_importance.csv"
    step3b_path = REPO_ROOT / "3a_feature_importance" / "outputs" / cohort_name / age_band_fname / "cohort_feature_importance.csv"
    
    print(f"{cohort_name}/{age_band}:")
    print(f"  Step 3b CSV: {'✓' if step3b_path.exists() else '✗'} {step3b_path}")
    print(f"  NB3 CSV:     {'✓' if combined_path.exists() else '✗'} {combined_path}")
    
    # Generate from Step 3b or notebook 3 combined_importance.csv
    try:
        if write_shap_ffa_allowed_codes_for_bupar(
            cohort_name, age_band, allowed_path, top_n=500, 
            project_root=REPO_ROOT, data_root=DATA_ROOT
        ):
            generated += 1
            print(f"  → ✓ Generated {allowed_path.name}")
        else:
            failed += 1
            print(f"  → ✗ No codes extracted (both sources missing or empty)")
    except Exception as e:
        failed += 1
        print(f"  → ✗ ERROR: {e}")
        import traceback
        traceback.print_exc()
    print()

print("="*80)
print(f"Summary: Generated={generated}, Skipped={skipped}, Failed={failed}, Total={len(combinations)}")
if failed > 0:
    print("⚠️  Some files failed to generate. Check paths above.")
print("="*80)
print()

# Prerequisite: SHAP/FFA combined allowed codes (required by BupaR and DTW; we never use all codes).
# These are built from either Step 3b feature importance or notebook 3 combined_importance.csv (fallback).
import json
missing = []
empty = []
for cohort_name, age_band in combinations:
    age_band_fname = age_band.replace("-", "_")
    path = BUPAR_OUTPUTS / f"allowed_codes_shap_ffa_{cohort_name}_{age_band_fname}.json"
    if not path.exists():
        missing.append(f"{cohort_name}/{age_band} ({path.name})")
    else:
        try:
            with open(path, encoding="utf-8") as f:
                codes = json.load(f)
            if not codes or (isinstance(codes, list) and len(codes) == 0):
                empty.append(f"{cohort_name}/{age_band} ({path.name})")
        except Exception as e:
            empty.append(f"{cohort_name}/{age_band} ({path.name}): {e}")
if missing or empty:
    msg = "SHAP/FFA combined allowed codes are required for BupaR and DTW (prerequisite).\n"
    if missing:
        msg += f"  Missing: {', '.join(missing)}\n"
    if empty:
        msg += f"  Empty or invalid: {', '.join(empty)}\n"
    msg += "  Sources (checked in order):\n"
    msg += "    1. Step 3b feature importance: 3a_feature_importance/outputs/{cohort}/{age_band}/cohort_feature_importance.csv\n"
    msg += "    2. Notebook 3 combined importance: 10_risk_dashboard/outputs/{cohort}/{age_band}/combined_importance.csv\n"
    msg += "  Generate allowed codes by running create_bupar_visuals.py with --write-allowed-codes, or sync from S3 gold/bupar/.\n"
    msg += "  If both sources above are missing, re-run feature importance (notebook 2) or SHAP/FFA combine (notebook 3)."
    raise RuntimeError(msg)
print(f"Prerequisite check passed: all {len(combinations)} SHAP/FFA combined allowed codes files present.")
print("  Sources: Step 3b feature importance (primary) or notebook 3 combined_importance.csv (fallback)")

Using pipeline-supported cohort/age_band (REQUIRED_COHORTS)
Cohorts: ['opioid_ed', 'non_opioid_ed']
Age bands: per-cohort (REQUIRED_COHORTS)
Total: 16 combinations

Generating allowed_codes JSON files from feature importance data...
Repo root: /home/pgx3874/pgx-analysis
Data root: /mnt/nvme
Looking for:
  - Step 3b: 3a_feature_importance/outputs/{cohort}/{age_band}/cohort_feature_importance.csv
  - Notebook 3: 10_risk_dashboard/outputs/{cohort}/{age_band}/combined_importance.csv

Summary: Generated=0, Skipped=16, Failed=0, Total=16

Prerequisite check passed: all 16 SHAP/FFA combined allowed codes files present.
  Sources: Step 3b feature importance (primary) or notebook 3 combined_importance.csv (fallback)


In [13]:
# (Prerequisite check runs in the Config cell above.)

## Feature importance heatmaps (dashboard Feature Importance tab)

Build aggregated and combined feature importance heatmaps from Step 3a outputs. **Saved locations** (used by notebook 5 and deploy sync):

- Per cohort: `3a_feature_importance/outputs/{cohort}/plots/{cohort}_aggregated_fi_heatmap.png`
- Combined: `3a_feature_importance/outputs/plots/combined_cohorts_feature_importance_heatmap.png`

Prerequisite: Step 3a aggregated CSVs at `3a_feature_importance/outputs/{cohort}/{cohort}_{age_band}_aggregated_feature_importance.csv`.

In [ ]:
# Build FI heatmaps for dashboard (saved where notebook 5 / deploy expect them)
from py_helpers.feature_importance_heatmap import create_aggregated_fi_heatmap, create_combined_cohorts_fi_heatmap

FI_OUTPUTS_BASE = REPO_ROOT / "3a_feature_importance" / "outputs"
# REQUIRED_COHORTS from Config cell: {cohort: [age_bands]}
paths_per_cohort = []
for cohort, age_bands in REQUIRED_COHORTS.items():
    p = create_aggregated_fi_heatmap(cohort, age_bands, FI_OUTPUTS_BASE, top_n=50)
    if p:
        paths_per_cohort.append(p)
        print(f"  ✓ {cohort}: {p}")
    else:
        print(f"  ✗ {cohort}: no aggregated CSVs found under {FI_OUTPUTS_BASE / cohort}")

combined_path = create_combined_cohorts_fi_heatmap(FI_OUTPUTS_BASE, REQUIRED_COHORTS, top_n=80)
if combined_path:
    print(f"  ✓ Combined: {combined_path}")
else:
    print("  ✗ Combined: no data (ensure at least one cohort has aggregated CSVs)")

print()
print("Save locations (must match notebook 5 requirement check and deploy sync):")
print(f"  Per cohort: {FI_OUTPUTS_BASE}/<cohort>/plots/<cohort>_aggregated_fi_heatmap.png")
print(f"  Combined:  {FI_OUTPUTS_BASE}/plots/combined_cohorts_feature_importance_heatmap.png")

## Run BupaR process mining

Plots are uploaded to the dashboard bucket under `{S3_DASHBOARD_PREFIX}/bupar/{cohort}/{age_band}/plots/` (same pattern as FP-Growth). **BupaR features are not used for feature engineering** (same as DTW and FP-Growth); they are computed for dashboard visualization and analysis.

In [ ]:
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

FAIL_FAST = True
FORCE_RERUN = True
force_flag = ["--force"] if FORCE_RERUN else []

def run_bupar_one(cohort_name, age_band):
    r = subprocess.run(
        [sys.executable, str(BUPAR_VISUALS_SCRIPT), "--cohort-name", cohort_name, "--age-band", age_band] + force_flag,
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
    )
    return (cohort_name, age_band, r.returncode, r.stdout, r.stderr)

with ThreadPoolExecutor(max_workers=PARALLEL_WORKERS) as ex:
    futures = {ex.submit(run_bupar_one, c, ab): (c, ab) for c, ab in combinations}
    for fut in as_completed(futures):
        cohort_name, age_band, code, stdout, stderr = fut.result()
        print(f"  [BupaR] {cohort_name} / {age_band} -> exit {code}")
        if code != 0:
            ab_f = age_band.replace("-", "_")
            print(f"    create_bupar_visuals failed (exit {code}). Check 9_dashboard_visuals/logs/bupaR/bupar_{cohort_name}_{ab_f}.log if available")
            if stderr:
                print("    stderr:", (stderr[:1500] + "..." if len(stderr) > 1500 else stderr))
            if stdout:
                print("    stdout:", (stdout[:800] + "..." if len(stdout) > 800 else stdout))
            if FAIL_FAST:
                raise RuntimeError(f"BupaR failed: {cohort_name} / {age_band}")
print("BupaR done.")

  [BupaR] opioid_ed / 0-12 -> exit 0
  [BupaR] non_opioid_ed / 75-84 -> exit 0
  [BupaR] non_opioid_ed / 85-114 -> exit 0
  [BupaR] non_opioid_ed / 65-74 -> exit 0
  [BupaR] non_opioid_ed / 55-64 -> exit 0
  [BupaR] non_opioid_ed / 45-54 -> exit 0
  [BupaR] non_opioid_ed / 13-24 -> exit 0
  [BupaR] non_opioid_ed / 25-44 -> exit 0


## Run DTW trajectory analysis (visualization and analysis)

**DTW trajectories and alignment are for visualization and analysis** (not feature engineering). This step:
1. **Extracts trajectories** from model_data filtered by SHAP/FFA important codes (same as BupaR/FP-Growth)
2. **Sequence alignment** - DTW distance computation to prototype trajectories to identify common patterns
3. **Creates visualizations** (trajectory cluster plots, routine vs no routine charts, sequence heatmaps)
4. **Uploads to dashboard bucket** under `{S3_DASHBOARD_PREFIX}/dtw/{cohort}/{age_band}/plots/`

**What's created:**
- `dtw_features_{cohort}_{age_band}.csv` - Trajectory data with alignment:
  - `seq_pattern_str`: Sequence of activity codes (e.g., "DRUG:Med_ICD:F1120_CPT:99213")
  - `dtw_min_distance`: Distance to nearest prototype (sequence alignment measure)
  - `admin_icd_event_count`: Count of administrative ICD codes (routine vs no routine)
  - `trajectory_length`, `trajectory_diversity`, `mean_days_between_events`: Temporal metrics
- `common_sequences.json` - Top sequence patterns identified via DTW alignment
- Trajectory cluster plots + sequence heatmaps (code×position patterns)
- `chart_data.json` (routine_comparison, high_risk_trajectories, target_pathways) for dashboard


**Runtime:** ~10-30 minutes per cohort/age_band (includes DTW distance matrix computation for sequence alignment)

**Research Question:** Addresses "Routine vs no routine appointments → outcomes" and **sequence-level patterns** (which drug/ICD/CPT sequences precede adverse events). **Date scope:** full pipeline (2016–2019).

In [ ]:
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

try:
    FAIL_FAST
except NameError:
    FAIL_FAST = True  # Stop on first failure; set False to continue (also in config/BupaR cell)
force_flag = ["--force"] if FORCE_RERUN else []

def run_dtw_one(cohort_name, age_band):
    """Run DTW trajectory extraction, alignment, and visualization (three-step process)."""
    # Step 1: Extract trajectories from model_data (filtered by SHAP/FFA)
    r_traj = subprocess.run(
        [sys.executable, str(DTW_TRAJECTORIES_SCRIPT), 
         "--cohort", cohort_name, "--age-band", age_band] + force_flag,
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
    )
    if r_traj.returncode != 0:
        return (cohort_name, age_band, "trajectories", r_traj.returncode, r_traj.stdout, r_traj.stderr)
    
    # Step 2: DTW alignment (compute distances to prototypes, identify common sequences)
    r_align = subprocess.run(
        [sys.executable, str(DTW_FEATURES_SCRIPT), 
         "--cohort", cohort_name, "--age-band", age_band] + force_flag,
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
    )
    if r_align.returncode != 0:
        return (cohort_name, age_band, "alignment", r_align.returncode, r_align.stdout, r_align.stderr)
    
    # Step 3: Create and publish visualizations
    r_vis = subprocess.run(
        [sys.executable, str(DTW_VISUALS_SCRIPT), "--cohort-name", cohort_name, "--age-band", age_band,
         "--project-root", str(REPO_ROOT)] + force_flag,
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
    )
    if r_vis.returncode != 0:
        return (cohort_name, age_band, "visuals", r_vis.returncode, r_vis.stdout, r_vis.stderr)
    
    return (cohort_name, age_band, "success", 0, "", "")

with ThreadPoolExecutor(max_workers=PARALLEL_WORKERS) as ex:
    futures = {ex.submit(run_dtw_one, c, ab): (c, ab) for c, ab in combinations}
    for fut in as_completed(futures):
        cohort_name, age_band, step, code, stdout, stderr = fut.result()
        if code == 0:
            print(f"  [DTW] {cohort_name} / {age_band} -> SUCCESS")
        else:
            print(f"  [DTW] {cohort_name} / {age_band} -> FAILED at {step} (exit {code})")
            if stderr:
                print(f"    stderr: {stderr[:1500]}{'...' if len(stderr) > 1500 else ''}")
            if FAIL_FAST:
                raise RuntimeError(f"DTW {step} failed: {cohort_name} / {age_band}")
print("DTW done (trajectories + alignment + visuals).")

### Appointments vs no appointments and extreme-density cohorts

**Research question (N1):** Is there a difference in outcomes for patients without routine appointments vs those with routine care? The DTW tab answers this via **Routine vs No Routine (Outcomes)** and **High-Risk vs Low-Risk Trajectories** (see above). These are shown over the **full pipeline (2016–2019)**, not a single year.

**Extreme-density cohorts** are high-utilizer patients (top ~5% by medical_code transaction density) split out so they do not dominate main models (see `docs/Step4_ModelData/README_model_data_and_extreme_split.md`). For **each cohort and age band**, running extract + DTW (and optionally BupaR) for the extreme-density subgroup lets you compare **routine vs no routine** (outcomes and trajectories) in the high-utilizer subgroup and how **extreme densities** and **extreme-density trajectories** differ across age bands and cohorts. By default the cell below uses the **same (cohort, age_band) combinations** as the main pipeline. Set `EXTREME_COMBINATIONS = []` to skip. Requires **Step 4** (model data) first.

In [ ]:
# Default: same (cohort, age_band) as main pipeline so we get routine vs no routine and
# extreme-density trajectories for every cohort and age band. Set to [] to skip.
EXTREME_COMBINATIONS = combinations  # from config cell above

EXTREME_EXTRACT_SCRIPT = STEP9_ROOT / "dtw" / "extract_extreme_density_cohort.py"
extreme_force_flag = ["--force"] if FORCE_RERUN else []

def run_extreme_one(cohort_name, age_band):
    r0 = subprocess.run(
        [sys.executable, str(EXTREME_EXTRACT_SCRIPT), "--cohort-name", cohort_name, "--age-band", age_band],
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
    )
    if r0.returncode != 0:
        return (cohort_name, age_band, r0.returncode, None, r0.stdout, r0.stderr, None, None)
    extreme_name = f"{cohort_name}_extreme_density"
    r2 = subprocess.run(
        [sys.executable, str(DTW_VISUALS_SCRIPT), "--cohort-name", extreme_name, "--age-band", age_band,
         "--project-root", str(REPO_ROOT)] + extreme_force_flag,
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
    )
    return (cohort_name, age_band, r0.returncode, r2.returncode, r0.stdout, r0.stderr, r2.stdout, r2.stderr)

if not EXTREME_COMBINATIONS:
    print("EXTREME_COMBINATIONS is empty; skipping extreme-density cohort extraction and DTW.")
else:
    from concurrent.futures import ThreadPoolExecutor, as_completed
    with ThreadPoolExecutor(max_workers=PARALLEL_WORKERS) as ex:
        futures = {ex.submit(run_extreme_one, c, ab): (c, ab) for c, ab in EXTREME_COMBINATIONS}
        for fut in as_completed(futures):
            result = fut.result()
            cohort_name, age_band = result[0], result[1]
            c0, c2 = result[2], result[3]
            print(f"  [Extreme] {cohort_name} / {age_band} -> extract={c0}, dtw_vis={c2}")
            if c0 != 0:
                print(f"    extract_extreme_density_cohort failed (exit {c0})")
                if len(result) > 5 and result[5]:
                    print("    stderr:", (result[5][:1500] + "..." if len(result[5]) > 1500 else result[5]))
                if FAIL_FAST:
                    raise RuntimeError(f"Extract extreme cohort failed: {cohort_name} / {age_band}")
            if c2 is not None and c2 != 0:
                print(f"    create_dtw_visuals failed (exit {c2})")
                if FAIL_FAST:
                    raise RuntimeError(f"DTW create_dtw_visuals failed: {cohort_name}_extreme_density / {age_band}")
    print(f"Done: extreme-density extract + DTW visuals for {len(EXTREME_COMBINATIONS)} combinations (parallel).")

## Run FP-Growth (itemsets, Plotly network HTML, S3 upload)

FP-Growth uses **SHAP/FFA-refined** model data: inputs come from `4_model_data` (built from Step 3b `cohort_feature_importance.csv`). **Item types included: drugs (`drug_name`), ICD diagnosis codes (`icd_code`), and CPT procedure codes (`cpt_code`)** (plus combined `medical_code`). For each cohort/age band this step: (1) ensures itemsets exist, (2) creates PNGs and **Plotly interactive network HTML**, (3) **uploads to the dashboard bucket** (e.g. `jerome-dixon.io`) under `{S3_DASHBOARD_PREFIX}/fpgrowth/{cohort}/{age_band}/plots/` (e.g. `vcu/pgx-risk-calculator/fpgrowth/...`). The dashboard then shows the **network plot for the user-selected cohort** via the `/visualizations/fpgrowth` API. **FP-Growth itemsets and rules are not used for feature engineering** (same as DTW and BupaR); they are computed for dashboard visualization and analysis.

Run the cell below in parallel (FPGROWTH_WORKERS at a time; builds itemsets, Plotly HTML, uploads to S3). **Exit 0** = itemsets/plots produced for that cohort/age_band; **exit 1** = no outputs (e.g. model_data missing or too few transactions). Logs: `9_dashboard_visuals/logs/fpgrowth/` and S3 `6_fpgrowth_log/{cohort}/{age_band}/`.

In [ ]:
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

try:
    FAIL_FAST
except NameError:
    FAIL_FAST = True
try:
    FPGROWTH_WORKERS
except NameError:
    # Maximize CPU utilization: use all cores up to number of combinations
    import os
    FPGROWTH_WORKERS = min(os.cpu_count() or 4, len(combinations))
force_flag = ["--force"] if FORCE_RERUN else []

def run_fpgrowth_one(cohort_name, age_band):
    r = subprocess.run(
        [sys.executable, str(FPGROWTH_VISUALS_SCRIPT), "--cohort-name", cohort_name, "--age-band", age_band] + force_flag,
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
    )
    return (cohort_name, age_band, r.returncode, r.stdout, r.stderr)

with ThreadPoolExecutor(max_workers=FPGROWTH_WORKERS) as ex:
    futures = {ex.submit(run_fpgrowth_one, c, ab): (c, ab) for c, ab in combinations}
    for fut in as_completed(futures):
        cohort_name, age_band, code, stdout, stderr = fut.result()
        print(f"  [FP-Growth] {cohort_name} / {age_band} -> exit {code}")
        if code != 0:
            ab_f = age_band.replace("-", "_")
            print(f"    No itemsets produced (exit 1). Check 9_dashboard_visuals/logs/fpgrowth/fpgrowth_{cohort_name}_{ab_f}.log or s3://pgx-repository/6_fpgrowth_log/{cohort_name}/{age_band}/")
            if stderr:
                print("    stderr:", (stderr[:1500] + "..." if len(stderr) > 1500 else stderr))
            if stdout:
                print("    stdout:", (stdout[:800] + "..." if len(stdout) > 800 else stdout))
            if FAIL_FAST:
                raise RuntimeError(f"FP-Growth failed: {cohort_name} / {age_band}")
print("FP-Growth done.")

## View visualization outputs

**Outputs are not in the repo** — they are created when you run the BupaR, FP-Growth, and DTW cells above. Paths: **`10_risk_dashboard/visualizations/bupar/outputs/`**, **`.../dtw/outputs/`**, **`.../fpgrowth/outputs/`** (each has `cohort;age_band;/plots/` with PNG and HTML). Run the cell below **after** those steps to list paths and preview the first PNG (and an "Open in browser" link for HTML). If you see "No outputs yet", run the pipeline cells above first.

In [ ]:
# Where outputs live and a sample preview (run after BupaR / FP-Growth / DTW cells)
from pathlib import Path
from IPython.display import display, Image, HTML, IFrame

def _first_plots_dir(base: Path, subdir: str):
    out = base / subdir / "outputs"
    if not out.exists():
        return None
    for cohort in out.iterdir():
        if not cohort.is_dir() or cohort.name.startswith("."):
            continue
        for age in cohort.iterdir():
            if not age.is_dir():
                continue
            plots = age / "plots"
            if plots.exists():
                return plots
    return None

base = REPO_ROOT / "10_risk_dashboard" / "visualizations"
print("Output root:", base)
print()

for name, subdir in [("BupaR", "bupar"), ("DTW", "dtw"), ("FP-Growth", "fpgrowth")]:
    plots_dir = _first_plots_dir(base, subdir)
    print(f"--- {name} ---")
    if not plots_dir:
        print(f"  No outputs yet. Run the \"Run {name}\" cell above, then re-run this cell to see visuals here.")
        print(f"  Path checked: {base / subdir / 'outputs'}")
        continue
    print(f"  Sample dir: {plots_dir}")
    pngs = sorted(plots_dir.glob("*.png"))
    htmls = sorted(plots_dir.glob("*.html"))
    print(f"  PNGs: {len(pngs)}, HTMLs: {len(htmls)}")
    if pngs:
        display(HTML(f"<b>{name} (first PNG)</b>"))
        display(Image(filename=str(pngs[0]), width=600))
    if htmls:
        display(HTML(f"<b>{name} (first HTML)</b>"))
        try:
            display(IFrame(src=str(htmls[0]), width=700, height=450))
        except Exception as e:
            pass
        # Local file IFrame often blocked; show path to open in browser
        display(HTML(f'<a href="file:///{htmls[0].resolve().as_posix()}" target="_blank">Open in browser</a>'))
    print()
print("All outputs: 10_risk_dashboard/visualizations/{bupar,dtw,fpgrowth}/outputs/<cohort>/<age_band>/plots/")

## PGx Patient Card Setup (Optional)

The dashboard includes a **PGx Patient Card** feature (Tab 2) that generates personalized pharmacogenomic cards from genetic variants. The Lambda function (`POST /pgx/card`) uses **CPIC gene-drug pairs** data to match variants to drugs requiring dosing modifications.

**Optional enhancement**: Add **PharmGKB VIP URLs** and **QR codes** to patient cards for richer gene information. Run the cells below to:
1. Fetch PharmGKB VIP gene data (uses current API)
2. Generate QR codes pointing to ClinPGx VIP pages
3. Build unified PGx database for Lambda integration

**Note**: The Lambda already works with CPIC data alone. This step adds PharmGKB integration for enhanced cards.

**Python migration (2026)**: Old R scripts (`PGx.Rmd`, `Build_PGx_Database.Rmd`) are deprecated due to PharmGKB API changes. See `pgx-patient-card/README_PYTHON.md` for details.

In [14]:
# PGx Patient Card Setup - Paths
PGX_CARD_DIR = REPO_ROOT / "pgx-patient-card"
PGX_DATA_DIR = PGX_CARD_DIR / "data"
PGX_QR_DIR = PGX_CARD_DIR / "qr_codes"

# Scripts
DOWNLOAD_CPIC_SCRIPT = PGX_CARD_DIR / "download_cpic_excel.py"
FETCH_PHARMGKB_SCRIPT = PGX_CARD_DIR / "fetch_pharmgkb_data.py"
GENERATE_QR_SCRIPT = PGX_CARD_DIR / "generate_pgx_qr_codes.py"
BUILD_DATABASE_SCRIPT = PGX_CARD_DIR / "build_pgx_database.py"

# Data files
CPIC_EXCEL = PGX_DATA_DIR / "cpic_gene-drug_pairs.xlsx"
VIP_JSON = PGX_DATA_DIR / "pharmgkb_vip_genes.json"
QR_MAPPINGS_JSON = PGX_DATA_DIR / "qr_code_mappings.json"
PGX_DATABASE_DIR = PGX_DATA_DIR / "pgx_database"

print(f"PGx card directory: {PGX_CARD_DIR}")
print(f"Data directory: {PGX_DATA_DIR}")
print(f"QR codes directory: {PGX_QR_DIR}")
print()
print("Files:")
print(f"  CPIC Excel: {CPIC_EXCEL} {'✓' if CPIC_EXCEL.exists() else '✗'}")
print(f"  VIP JSON: {VIP_JSON} {'✓' if VIP_JSON.exists() else '✗'}")
print(f"  QR mappings: {QR_MAPPINGS_JSON} {'✓' if QR_MAPPINGS_JSON.exists() else '✗'}")
print(f"  Database: {PGX_DATABASE_DIR} {'✓' if PGX_DATABASE_DIR.exists() else '✗'}")

PGx card directory: /home/pgx3874/pgx-analysis/pgx-patient-card
Data directory: /home/pgx3874/pgx-analysis/pgx-patient-card/data
QR codes directory: /home/pgx3874/pgx-analysis/pgx-patient-card/qr_codes

Files:
  CPIC Excel: /home/pgx3874/pgx-analysis/pgx-patient-card/data/cpic_gene-drug_pairs.xlsx ✗
  VIP JSON: /home/pgx3874/pgx-analysis/pgx-patient-card/data/pharmgkb_vip_genes.json ✗
  QR mappings: /home/pgx3874/pgx-analysis/pgx-patient-card/data/qr_code_mappings.json ✗
  Database: /home/pgx3874/pgx-analysis/pgx-patient-card/data/pgx_database ✗


### Step 1: Download CPIC Data

Download the latest CPIC gene-drug pairs Excel file. This is the **primary data source** for the PGx card feature.

**URL**: `https://files.cpicpgx.org/data/report/current/pair/cpic_gene-drug_pairs.xlsx`

**Verified**: February 2026 (31KB, last modified Feb 5, 2026)

In [15]:
# Download CPIC gene-drug pairs Excel file (uses download_cpic_excel.py with SSL fallback)
if CPIC_EXCEL.exists() and CPIC_EXCEL.stat().st_size > 0:
    print(f"✓ CPIC Excel already exists: {CPIC_EXCEL}")
    print(f"  Size: {CPIC_EXCEL.stat().st_size / 1024:.1f} KB")
else:
    result = subprocess.run(
        [sys.executable, str(DOWNLOAD_CPIC_SCRIPT)],
        cwd=str(PGX_CARD_DIR),
        capture_output=True,
        text=True,
    )
    if result.returncode == 0:
        if result.stdout:
            print(result.stdout)
    else:
        print(result.stdout or "")
        print(result.stderr or "")
        raise RuntimeError(f"CPIC download failed (exit {result.returncode})")

✓ Downloaded (no verify): /home/pgx3874/pgx-analysis/pgx-patient-card/data/cpic_gene-drug_pairs.xlsx (30.4 KB)



### Step 2: Fetch PharmGKB VIP Data

Fetch VIP (Very Important Pharmacogene) data from PharmGKB API. This adds **ClinPGx VIP URLs** for genes.

**API**: `https://api.pharmgkb.org/v1/data/gene?symbol=...` (documented in Postman)

**Output**: `data/pharmgkb_vip_genes.json` with VIP URLs for 20 genes

In [16]:
# Fetch PharmGKB VIP gene data
if VIP_JSON.exists():
    print(f"✓ VIP data already exists: {VIP_JSON}")
    import json
    with open(VIP_JSON) as f:
        vip_data = json.load(f)
    print(f"  Contains {len(vip_data)} VIP genes")
else:
    print("Fetching PharmGKB VIP gene data...")
    print(f"  Running: {FETCH_PHARMGKB_SCRIPT}")
    result = subprocess.run(
        [sys.executable, str(FETCH_PHARMGKB_SCRIPT)],
        cwd=str(PGX_CARD_DIR),
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        print("✓ VIP data fetched successfully")
        print(result.stdout)
    else:
        print(f"✗ Failed (exit {result.returncode})")
        print(result.stderr)

Fetching PharmGKB VIP gene data...
  Running: /home/pgx3874/pgx-analysis/pgx-patient-card/fetch_pharmgkb_data.py
✓ VIP data fetched successfully
Fetching 20 VIP genes from PharmGKB API...
Saved 20 VIP genes to /home/pgx3874/pgx-analysis/pgx-patient-card/data/pharmgkb_vip_genes.json

Sample VIP gene:
{
  "gene": "CYP2C19",
  "gene_id": "PA124",
  "vip_url": "https://www.clinpgx.org/vip/PA124/overview",
  "qr_filename": "CYP2C19.png",
  "summary": "cytochrome P450 family 2 subfamily C member 19",
  "chromosome": ""
}



### Step 3: Generate QR Codes

Generate QR codes for ClinPGx VIP pages. Each gene gets a QR code pointing to `https://www.clinpgx.org/vip/{PA_ID}/overview`.

**Prerequisites**: `pip install qrcode[pil]`

**Output**: `qr_codes/{GENE}.png` (20 QR code images, 200x200 px)

In [17]:
# Generate QR codes for VIP pages
if QR_MAPPINGS_JSON.exists() and PGX_QR_DIR.exists() and len(list(PGX_QR_DIR.glob("*.png"))) > 0:
    print(f"✓ QR codes already exist: {PGX_QR_DIR}")
    print(f"  Contains {len(list(PGX_QR_DIR.glob('*.png')))} QR code images")
else:
    print("Generating QR codes for VIP pages...")
    print(f"  Running: {GENERATE_QR_SCRIPT}")
    result = subprocess.run(
        [sys.executable, str(GENERATE_QR_SCRIPT)],
        cwd=str(PGX_CARD_DIR),
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        print("✓ QR codes generated successfully")
        print(result.stdout)
    else:
        print(f"✗ Failed (exit {result.returncode})")
        print(result.stderr)
        if "ModuleNotFoundError" in result.stderr and "qrcode" in result.stderr:
            print("\nInstall missing dependency:")
            print("  pip install qrcode[pil]")

Generating QR codes for VIP pages...
  Running: /home/pgx3874/pgx-analysis/pgx-patient-card/generate_pgx_qr_codes.py
✓ QR codes generated successfully
Loading VIP gene data...
Found 20 VIP genes

Generating QR codes...
✓ Generated QR code for CYP2C19 -> CYP2C19.png
✓ Generated QR code for CYP2C9 -> CYP2C9.png
✓ Generated QR code for CYP2D6 -> CYP2D6.png
✓ Generated QR code for CYP3A5 -> CYP3A5.png
✓ Generated QR code for SLCO1B1 -> SLCO1B1.png
✓ Generated QR code for DPYD -> DPYD.png
✓ Generated QR code for TPMT -> TPMT.png
✓ Generated QR code for UGT1A1 -> UGT1A1.png
✓ Generated QR code for VKORC1 -> VKORC1.png
✓ Generated QR code for CFTR -> CFTR.png
✓ Generated QR code for G6PD -> G6PD.png
✓ Generated QR code for HLA-A -> HLA-A.png
✓ Generated QR code for HLA-B -> HLA-B.png
✓ Generated QR code for IFNL3 -> IFNL3.png
✓ Generated QR code for NUDT15 -> NUDT15.png
✓ Generated QR code for CYP4F2 -> CYP4F2.png
✓ Generated QR code for F5 -> F5.png
✓ Generated QR code for CACNA1S -> CACNA1S

### Step 4: Build Unified PGx Database

Merge CPIC, PharmGKB VIP, and QR code data into a unified database for patient card generation.

**Output**:
- `data/pgx_database/pgx_database.csv` - Merged data (CSV)
- `data/pgx_database/pgx_database.json` - Merged data (JSON)
- `data/pgx_database/pgx_database.xlsx` - Merged data (Excel)
- `data/pgx_database/pgx_database_summary.json` - Summary statistics

In [18]:
# Build unified PGx database
if PGX_DATABASE_DIR.exists() and (PGX_DATABASE_DIR / "pgx_database.csv").exists():
    print(f"✓ PGx database already exists: {PGX_DATABASE_DIR}")
    import json
    summary_path = PGX_DATABASE_DIR / "pgx_database_summary.json"
    if summary_path.exists():
        with open(summary_path) as f:
            summary = json.load(f)
        print("\nDatabase summary:")
        for key, value in summary.items():
            print(f"  {key.replace('_', ' ').title()}: {value}")
else:
    print("Building unified PGx database...")
    print(f"  Running: {BUILD_DATABASE_SCRIPT}")
    result = subprocess.run(
        [sys.executable, str(BUILD_DATABASE_SCRIPT)],
        cwd=str(PGX_CARD_DIR),
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        print("✓ PGx database built successfully")
        print(result.stdout)
    else:
        print(f"✗ Failed (exit {result.returncode})")
        print(result.stderr)

Building unified PGx database...
  Running: /home/pgx3874/pgx-analysis/pgx-patient-card/build_pgx_database.py
✓ PGx database built successfully
Building PGx patient database...

1. Loading CPIC data...
Loaded 573 gene-drug pairs from CPIC

2. Loading PharmGKB VIP data...
Loaded 20 VIP genes from PharmGKB

3. Loading QR code mappings...
Loaded 20 QR code mappings

4. Merging databases...

Merged database: 573 rows
Unique genes: 121
Unique drugs: 300

5. Saving outputs...
✓ Saved CSV: /home/pgx3874/pgx-analysis/pgx-patient-card/data/pgx_database/pgx_database.csv
✓ Saved JSON: /home/pgx3874/pgx-analysis/pgx-patient-card/data/pgx_database/pgx_database.json
✓ Saved Excel: /home/pgx3874/pgx-analysis/pgx-patient-card/data/pgx_database/pgx_database.xlsx
✓ Saved summary: /home/pgx3874/pgx-analysis/pgx-patient-card/data/pgx_database/pgx_database_summary.json

PGx Database Summary:
  Total Rows: 573
  Unique Genes: 121
  Unique Drugs: 300
  Genes With Vip Url: 266
  Genes With Qr Code: 266



### Integration with Lambda

To integrate VIP URLs with the Lambda function (`POST /pgx/card`):

1. **Copy VIP data to Lambda container** (in Dockerfile or build script):
   ```dockerfile
   COPY pgx-patient-card/data/pharmgkb_vip_genes.json ${LAMBDA_TASK_ROOT}/data/
   ```

2. **Load at Lambda startup** (in `lambda_function.py`):
   ```python
   VIP_URL_CACHE = {}
   
   def load_vip_urls():
       vip_path = '/var/task/data/pharmgkb_vip_genes.json'
       if os.path.exists(vip_path):
           with open(vip_path) as f:
               vip_data = json.load(f)
           VIP_URL_CACHE = {item['gene'].upper(): item['vip_url'] for item in vip_data}
   
   load_vip_urls()  # Call at module level
   ```

3. **Add VIP URLs to card response** (in `generate_pgx_card()`):
   ```python
   for gene_entry in genes_processed:
       if gene_entry['gene'] in VIP_URL_CACHE:
           gene_entry['vip_url'] = VIP_URL_CACHE[gene_entry['gene']]
   ```

See `pgx-patient-card/README_PYTHON.md` for full Lambda integration details.

## Cohort PGx Network Topology

The dashboard includes a **Cohort PGx** tab that combines PharmGKB VIP reports for all important genes in a cohort with network topology analysis. This feature:

1. **Extracts PGx genes** from SHAP/FFA feature importance (top N genes from Step 3b or notebook 3)
2. **Fetches PharmGKB VIP reports** with clinical annotations and drug interactions
3. **Builds network topology** using:
   - **pytextrank**: Key phrase extraction and entity recognition from VIP text
   - **AWS Comprehend** (optional): Medical entity recognition and key phrase extraction
   - **Network analysis**: Gene-drug-phenotype relationships visualized as Plotly interactive graph

**Output**: Interactive network visualization showing how cohort-specific genes relate to drugs and phenotypes, plus exportable node/edge data for further analysis.

**Research value**: Identifies gene-drug interaction patterns specific to each cohort/age band, revealing pharmacogenomic mechanisms underlying adverse event risk.

In [19]:
# Cohort PGx Network Topology - Paths
COHORT_PGX_DIR = STEP9_ROOT / "cohort_pgx"
COHORT_PGX_REPORTS_DIR = VISUAL_ROOT / "cohort_pgx" / "reports"
COHORT_PGX_NETWORKS_DIR = VISUAL_ROOT / "cohort_pgx" / "networks"

# Scripts
FETCH_VIP_REPORTS_SCRIPT = COHORT_PGX_DIR / "fetch_vip_reports.py"
BUILD_NETWORK_TOPOLOGY_SCRIPT = COHORT_PGX_DIR / "build_network_topology.py"

print(f"Cohort PGx directory: {COHORT_PGX_DIR}")
print(f"Scripts:")
print(f"  Fetch VIP reports: {FETCH_VIP_REPORTS_SCRIPT} {'✓' if FETCH_VIP_REPORTS_SCRIPT.exists() else '✗'}")
print(f"  Build network: {BUILD_NETWORK_TOPOLOGY_SCRIPT} {'✓' if BUILD_NETWORK_TOPOLOGY_SCRIPT.exists() else '✗'}")
print()
print(f"Outputs:")
print(f"  Reports: {COHORT_PGX_REPORTS_DIR}")
print(f"  Networks: {COHORT_PGX_NETWORKS_DIR}")

# Config
COHORT_PGX_TOP_N = 50  # Top N genes from feature importance
COHORT_PGX_COHORTS = combinations  # Use same cohort/age_band combinations as main pipeline
USE_COMPREHEND = True  # Set False to skip AWS Comprehend (pytextrank only)

print()
print(f"Config:")
print(f"  Top N genes per cohort: {COHORT_PGX_TOP_N}")
print(f"  Cohort combinations: {len(COHORT_PGX_COHORTS)}")
print(f"  AWS Comprehend: {'Enabled' if USE_COMPREHEND else 'Disabled (pytextrank only)'}")

Cohort PGx directory: /home/pgx3874/pgx-analysis/9_dashboard_visuals/cohort_pgx
Scripts:
  Fetch VIP reports: /home/pgx3874/pgx-analysis/9_dashboard_visuals/cohort_pgx/fetch_vip_reports.py ✓
  Build network: /home/pgx3874/pgx-analysis/9_dashboard_visuals/cohort_pgx/build_network_topology.py ✓

Outputs:
  Reports: /home/pgx3874/pgx-analysis/10_risk_dashboard/visualizations/cohort_pgx/reports
  Networks: /home/pgx3874/pgx-analysis/10_risk_dashboard/visualizations/cohort_pgx/networks

Config:
  Top N genes per cohort: 50
  Cohort combinations: 16
  AWS Comprehend: Enabled


### Step 1: Fetch PharmGKB VIP Reports

For each cohort/age band, extract top N genes from feature importance and fetch VIP reports from PharmGKB API.

**Prerequisites**:
- Feature importance data (Step 3b or notebook 3 combined_importance.csv)
- `pip install beautifulsoup4` for VIP page text extraction

**Output**: `{cohort}_{age_band}_vip_reports.json` with gene metadata, clinical annotations, and VIP page text

**Runtime**: ~1-2 minutes per cohort/age band (API rate limited to 0.5s between requests)

In [20]:
# Fetch VIP reports for all cohort/age_band combinations
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

def run_fetch_vip_reports(cohort_name, age_band):
    """Fetch PharmGKB VIP reports for one cohort/age band."""
    args = [
        sys.executable, str(FETCH_VIP_REPORTS_SCRIPT),
        "--cohort", cohort_name,
        "--age-band", age_band,
        "--top-n", str(COHORT_PGX_TOP_N),
        "--project-root", str(REPO_ROOT),
        "--output-dir", str(COHORT_PGX_REPORTS_DIR)
    ]
    
    # Skip VIP page fetching if needed (faster but less text data)
    # args.append("--no-vip-pages")
    
    result = subprocess.run(args, cwd=str(REPO_ROOT), capture_output=True, text=True)
    return (cohort_name, age_band, result.returncode, result.stdout, result.stderr)

print(f"Fetching VIP reports for {len(COHORT_PGX_COHORTS)} cohort/age_band combinations...")
print(f"  Top {COHORT_PGX_TOP_N} genes per cohort")
print()

# Run in parallel (max 2 at a time to respect API rate limits)
MAX_WORKERS_PGX = 2  # Conservative to avoid API throttling

with ThreadPoolExecutor(max_workers=MAX_WORKERS_PGX) as ex:
    futures = {ex.submit(run_fetch_vip_reports, c, ab): (c, ab) for c, ab in COHORT_PGX_COHORTS}
    for fut in as_completed(futures):
        cohort_name, age_band, code, stdout, stderr = fut.result()
        if code == 0:
            print(f"  [VIP Reports] {cohort_name} / {age_band} -> SUCCESS")
        else:
            print(f"  [VIP Reports] {cohort_name} / {age_band} -> FAILED (exit {code})")
            if stderr:
                print(f"    stderr: {stderr[:800]}{'...' if len(stderr) > 800 else ''}")
            if FAIL_FAST:
                raise RuntimeError(f"Fetch VIP reports failed: {cohort_name} / {age_band}")

print("\n✓ VIP reports fetched for all cohorts")
print(f"  Output: {COHORT_PGX_REPORTS_DIR}")

Fetching VIP reports for 16 cohort/age_band combinations...
  Top 50 genes per cohort

  [VIP Reports] opioid_ed / 0-12 -> SUCCESS
  [VIP Reports] opioid_ed / 13-24 -> SUCCESS
  [VIP Reports] opioid_ed / 25-44 -> SUCCESS
  [VIP Reports] opioid_ed / 45-54 -> SUCCESS
  [VIP Reports] opioid_ed / 55-64 -> SUCCESS
  [VIP Reports] opioid_ed / 65-74 -> SUCCESS
  [VIP Reports] opioid_ed / 75-84 -> SUCCESS
  [VIP Reports] opioid_ed / 85-114 -> SUCCESS
  [VIP Reports] non_opioid_ed / 0-12 -> SUCCESS
  [VIP Reports] non_opioid_ed / 13-24 -> SUCCESS
  [VIP Reports] non_opioid_ed / 25-44 -> SUCCESS
  [VIP Reports] non_opioid_ed / 45-54 -> SUCCESS
  [VIP Reports] non_opioid_ed / 55-64 -> SUCCESS
  [VIP Reports] non_opioid_ed / 65-74 -> SUCCESS
  [VIP Reports] non_opioid_ed / 75-84 -> SUCCESS
  [VIP Reports] non_opioid_ed / 85-114 -> SUCCESS

✓ VIP reports fetched for all cohorts
  Output: /home/pgx3874/pgx-analysis/10_risk_dashboard/visualizations/cohort_pgx/reports


### Step 2: Build Network Topology

Build interactive network topology from VIP reports using pytextrank and AWS Comprehend.

**Prerequisites**:
- `pip install spacy pytextrank networkx plotly`
- `python -m spacy download en_core_web_sm` (spaCy model)
- `pip install boto3` (optional, for AWS Comprehend)

**Output per cohort/age band**:
- `network_topology.html` - Interactive Plotly network visualization
- `network_nodes.csv` - Node data (genes, drugs, phenotypes)
- `network_edges.csv` - Edge data (relationships and weights)
- `key_phrases.json` - Extracted key phrases per gene
- `network_stats.json` - Network statistics (density, degree, etc.)

**Runtime**: ~2-5 minutes per cohort/age band (depends on text volume and Comprehend usage)

In [22]:
# Build network topology for all cohort/age_band combinations
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

def run_build_network(cohort_name, age_band):
    """Build network topology for one cohort/age band."""
    age_band_fname = age_band.replace("-", "_")
    reports_file = COHORT_PGX_REPORTS_DIR / f"{cohort_name}_{age_band_fname}_vip_reports.json"
    output_dir = COHORT_PGX_NETWORKS_DIR / cohort_name / age_band_fname
    
    if not reports_file.exists():
        return (cohort_name, age_band, -1, f"Reports file not found: {reports_file}", "")
    
    args = [
        sys.executable, str(BUILD_NETWORK_TOPOLOGY_SCRIPT),
        "--reports", str(reports_file),
        "--output-dir", str(output_dir)
    ]
    
    if not USE_COMPREHEND:
        args.append("--no-comprehend")
    
    result = subprocess.run(args, cwd=str(REPO_ROOT), capture_output=True, text=True)
    return (cohort_name, age_band, result.returncode, result.stdout, result.stderr)

print(f"Building network topology for {len(COHORT_PGX_COHORTS)} cohort/age_band combinations...")
print(f"  AWS Comprehend: {'Enabled' if USE_COMPREHEND else 'Disabled'}")
print()

# Run in parallel (max 4 at a time)
MAX_WORKERS_NETWORK = 4

with ThreadPoolExecutor(max_workers=MAX_WORKERS_NETWORK) as ex:
    futures = {ex.submit(run_build_network, c, ab): (c, ab) for c, ab in COHORT_PGX_COHORTS}
    for fut in as_completed(futures):
        cohort_name, age_band, code, stdout, stderr = fut.result()
        if code == 0:
            print(f"  [Network Topology] {cohort_name} / {age_band} -> SUCCESS")
        else:
            print(f"  [Network Topology] {cohort_name} / {age_band} -> FAILED (exit {code})")
            if stderr:
                print(f"    stderr: {stderr[:800]}{'...' if len(stderr) > 800 else ''}")
            if stdout and "not found" in stdout.lower():
                print(f"    {stdout[:400]}{'...' if len(stdout) > 400 else ''}")
            if FAIL_FAST:
                raise RuntimeError(f"Build network topology failed: {cohort_name} / {age_band}")

print("\n✓ Network topology built for all cohorts")
print(f"  Output: {COHORT_PGX_NETWORKS_DIR}")

Building network topology for 16 cohort/age_band combinations...
  AWS Comprehend: Enabled

  [Network Topology] opioid_ed / 25-44 -> SUCCESS
  [Network Topology] opioid_ed / 45-54 -> SUCCESS
  [Network Topology] opioid_ed / 0-12 -> SUCCESS
  [Network Topology] opioid_ed / 13-24 -> SUCCESS
  [Network Topology] opioid_ed / 85-114 -> SUCCESS
  [Network Topology] opioid_ed / 75-84 -> SUCCESS
  [Network Topology] opioid_ed / 55-64 -> SUCCESS
  [Network Topology] opioid_ed / 65-74 -> SUCCESS
  [Network Topology] non_opioid_ed / 25-44 -> SUCCESS
  [Network Topology] non_opioid_ed / 13-24 -> SUCCESS
  [Network Topology] non_opioid_ed / 45-54 -> SUCCESS
  [Network Topology] non_opioid_ed / 0-12 -> SUCCESS
  [Network Topology] non_opioid_ed / 65-74 -> SUCCESS
  [Network Topology] non_opioid_ed / 55-64 -> SUCCESS
  [Network Topology] non_opioid_ed / 75-84 -> SUCCESS
  [Network Topology] non_opioid_ed / 85-114 -> SUCCESS

✓ Network topology built for all cohorts
  Output: /home/pgx3874/pgx-analys

### View Cohort PGx Network Outputs

View network topology visualizations and statistics for all cohorts.

In [23]:
# View Cohort PGx network outputs
from pathlib import Path
from IPython.display import display, HTML, IFrame
import json

print(f"Cohort PGx Network Outputs: {COHORT_PGX_NETWORKS_DIR}")
print()

if not COHORT_PGX_NETWORKS_DIR.exists():
    print("✗ No outputs yet. Run the cells above to fetch VIP reports and build networks.")
else:
    # Find all network topology HTML files
    html_files = sorted(COHORT_PGX_NETWORKS_DIR.glob("*/*/network_topology.html"))
    
    if not html_files:
        print("✗ No network topology files found. Check if build step completed successfully.")
    else:
        print(f"Found {len(html_files)} network topology visualizations\n")
        
        # Show first one as example
        first_html = html_files[0]
        cohort_name = first_html.parent.parent.name
        age_band = first_html.parent.name.replace("_", "-")
        
        print(f"Example: {cohort_name} / {age_band}")
        print(f"  HTML: {first_html}")
        
        # Show statistics
        stats_file = first_html.parent / "network_stats.json"
        if stats_file.exists():
            with open(stats_file) as f:
                stats = json.load(f)
            print(f"\nNetwork Statistics:")
            for key, value in stats.items():
                print(f"  {key.replace('_', ' ').title()}: {value}")
        
        print(f"\nInteractive visualization:")
        display(HTML(f'<a href="file:///{first_html.resolve().as_posix()}" target="_blank">Open {cohort_name}/{age_band} network in browser</a>'))
        
        # List all outputs
        print(f"\nAll network outputs:")
        for html_file in html_files:
            cohort = html_file.parent.parent.name
            age = html_file.parent.name.replace("_", "-")
            print(f"  - {cohort} / {age}: {html_file.parent}")

print(f"\nDashboard integration: Upload network_topology.html files to S3 dashboard bucket")
print(f"  S3 path: {{S3_DASHBOARD_PREFIX}}/cohort_pgx/{{cohort}}/{{age_band}}/")

Cohort PGx Network Outputs: /home/pgx3874/pgx-analysis/10_risk_dashboard/visualizations/cohort_pgx/networks

Found 16 network topology visualizations

Example: non_opioid_ed / 0-12
  HTML: /home/pgx3874/pgx-analysis/10_risk_dashboard/visualizations/cohort_pgx/networks/non_opioid_ed/0_12/network_topology.html

Network Statistics:
  Nodes Total: 0
  Edges Total: 0
  Genes: 0
  Drugs: 0
  Phenotypes: 0
  Cpic Genes: 0
  Drug Drug Interactions: 0
  Density: 0
  Avg Degree: 0
  Gene Tiers: {}

Interactive visualization:



All network outputs:
  - non_opioid_ed / 0-12: /home/pgx3874/pgx-analysis/10_risk_dashboard/visualizations/cohort_pgx/networks/non_opioid_ed/0_12
  - non_opioid_ed / 13-24: /home/pgx3874/pgx-analysis/10_risk_dashboard/visualizations/cohort_pgx/networks/non_opioid_ed/13_24
  - non_opioid_ed / 25-44: /home/pgx3874/pgx-analysis/10_risk_dashboard/visualizations/cohort_pgx/networks/non_opioid_ed/25_44
  - non_opioid_ed / 45-54: /home/pgx3874/pgx-analysis/10_risk_dashboard/visualizations/cohort_pgx/networks/non_opioid_ed/45_54
  - non_opioid_ed / 55-64: /home/pgx3874/pgx-analysis/10_risk_dashboard/visualizations/cohort_pgx/networks/non_opioid_ed/55_64
  - non_opioid_ed / 65-74: /home/pgx3874/pgx-analysis/10_risk_dashboard/visualizations/cohort_pgx/networks/non_opioid_ed/65_74
  - non_opioid_ed / 75-84: /home/pgx3874/pgx-analysis/10_risk_dashboard/visualizations/cohort_pgx/networks/non_opioid_ed/75_84
  - non_opioid_ed / 85-114: /home/pgx3874/pgx-analysis/10_risk_dashboard/visualizations/coh

### Dashboard Integration

To integrate Cohort PGx network topology into the dashboard:

1. **Upload to S3** (in notebook 5 or deploy script):
   ```bash
   aws s3 sync 10_risk_dashboard/visualizations/cohort_pgx/ \
     s3://{{DASHBOARD_BUCKET}}/{{S3_PREFIX}}/cohort_pgx/ \
     --exclude "*.csv" --exclude "*.json" --include "*.html"
   ```

2. **Add API endpoint** (in Lambda `lambda_function.py`):
   ```python
   @app.get("/visualizations/cohort-pgx")
   def get_cohort_pgx_viz(cohort: str, age_band: str):
       age_band_fname = age_band.replace("-", "_")
       base_url = f"https://{DASHBOARD_BUCKET}/{S3_PREFIX}/cohort_pgx/{cohort}/{age_band_fname}"
       return {
           "network_topology": f"{base_url}/network_topology.html",
           "network_nodes": f"{base_url}/network_nodes.csv",
           "network_edges": f"{base_url}/network_edges.csv",
           "network_stats": f"{base_url}/network_stats.json"
       }
   ```

3. **Add dashboard tab** (in frontend `index.html`):
   - Create new tab "Cohort PGx" after "PGx Card"
   - Load network topology HTML via iframe from API endpoint
   - Show network statistics in sidebar

See `10_risk_dashboard/docs/` for full integration guide.

## API (reference)

Lambda receives **user input** (cohort, age_band, model/feature selections) and **filters** only—it does not process or generate visualization data. All BupaR, DTW, and FP-Growth visuals are **prebuilt on EC2** and **saved to S3**; the API returns **URLs** to those prebuilt assets (filtered by cohort/age_band). Endpoints: `GET /visualizations/causal`, `/visualizations/bupar`, `/visualizations/dtw`, `/visualizations/fpgrowth`. See `10_risk_dashboard/backend/README.md`.

In [24]:
print("Dashboard endpoints: 10_risk_dashboard/backend/README.md")
print("API Gateway deploy: utility_scripts/create_api_gateway_pgx_risk_calculator.sh")

Dashboard endpoints: 10_risk_dashboard/backend/README.md
API Gateway deploy: utility_scripts/create_api_gateway_pgx_risk_calculator.sh


## Next: Build and deploy

Build and deploy run **only** in [5_build_and_deploy.ipynb](5_build_and_deploy.ipynb). Run that notebook after this one.

In [25]:
# Build and deploy run only in 5_build_and_deploy.ipynb. Run that notebook after this one.
print("Build and deploy (once): open 5_build_and_deploy.ipynb and run it after this notebook.")

Build and deploy (once): open 5_build_and_deploy.ipynb and run it after this notebook.


*(Build and deploy — including frontend sync to S3 — are done only in notebook 5. See above.)*